In [1]:
from pyspark.sql import SparkSession

spark = SparkSession. \
builder. \
appName("week6-lesson-3"). \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/itv024128/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [2]:
# You have a hospital dataset with the following fields:
#     patient_id (integer): Unique identifier for each patient.
#     admission_date (date): The date the patient was admitted to thehospital. (MM-dd-yyyy)
#     discharge_date (date): The date the patient was discharged from thehospital. (yyyy-MM-dd) 
#     diagnosis (string): The diagnosed medical condition of the patient.
#     doctor_id (integer): The identifier of the doctor responsible for thepatient's care.
#     total_cost (float): The total cost of the hospital stay for the patient.
# Using PySpark, load the data into a Dataframe and perform the followingoperations on the hospital dataset(/public/trendytech/datasets/hospital.csv):

In [3]:
!hadoop fs -head /public/trendytech/datasets/hospital.csv

patient_id,admission_date,discharge_date,diagnosis,doctor_id,total_cost
1,01-01-2022,2022-01-10,Pneumonia,101,5000.00
2,02-05-2022,2022-02-09,Appendicitis,102,7000.00
3,03-12-2022,2022-03-18,Fractured Arm,103,3500.00
4,04-02-2022,2022-04-08,Heart Attack,104,15000.00
5,05-05-2022,2022-05-07,Influenza,105,2500.00
6,06-10-2022,2022-06-15,Appendicitis,106,8000.00
7,07-20-2022,2022-07-25,Pneumonia,107,5500.00
8,08-25-2022,2022-09-01,Heart Attack,108,20000.00
9,09-15-2022,2022-09-22,Fractured Leg,109,6000.00
10,10-05-2022,2022-10-10,Appendicitis,110,7500.00
11,11-02-2022,2022-11-05,Influenza,111,2800.00
12,12-10-2022,2022-12-18,Pneumonia,112,6000.00
13,01-02-2023,2023-01-09,Heart Attack,113,18000.00
14,02-14-2023,2023-02-18,Appendicitis,114,7200.00
15,03-20-2023,2023-03-28,Fractured Arm,115,3800.00
16,04-05-2023,2023-04-11,Influenza,116,2700.00
17,05-08-2023,2023-05-11,Heart Attack,117,16000.00
18,06-15-2023,2023-06-20,Pneumonia,118,4800.00
19,07-22-2023,2023-07-27,Fractured Leg,119,6500.00


In [4]:
hosp_schema = 'patient_id int ,admission_date string  ,discharge_date string ,diagnosis string ,doctor_id long ,total_cost double'

In [5]:
hosp_df = spark.read.schema(hosp_schema).option('header','true').csv('/public/trendytech/datasets/hospital.csv')

In [6]:
hosp_df.show(4)

+----------+--------------+--------------+-------------+---------+----------+
|patient_id|admission_date|discharge_date|    diagnosis|doctor_id|total_cost|
+----------+--------------+--------------+-------------+---------+----------+
|         1|    01-01-2022|    2022-01-10|    Pneumonia|      101|    5000.0|
|         2|    02-05-2022|    2022-02-09| Appendicitis|      102|    7000.0|
|         3|    03-12-2022|    2022-03-18|Fractured Arm|      103|    3500.0|
|         4|    04-02-2022|    2022-04-08| Heart Attack|      104|   15000.0|
+----------+--------------+--------------+-------------+---------+----------+
only showing top 4 rows



In [7]:
hosp_df.count()

25

In [8]:
hosp_df.printSchema()

root
 |-- patient_id: integer (nullable = true)
 |-- admission_date: string (nullable = true)
 |-- discharge_date: string (nullable = true)
 |-- diagnosis: string (nullable = true)
 |-- doctor_id: long (nullable = true)
 |-- total_cost: double (nullable = true)



In [9]:
hosp_df1 = hosp_df.drop("doctor_id")

In [11]:
hosp_df1.show(2)

+----------+--------------+--------------+------------+----------+
|patient_id|admission_date|discharge_date|   diagnosis|total_cost|
+----------+--------------+--------------+------------+----------+
|         1|    01-01-2022|    2022-01-10|   Pneumonia|    5000.0|
|         2|    02-05-2022|    2022-02-09|Appendicitis|    7000.0|
+----------+--------------+--------------+------------+----------+
only showing top 2 rows



In [12]:
hosp_df2 = hosp_df1.withColumnRenamed("total_cost","hospital_bill")

In [13]:
hosp_df2.show(2)

+----------+--------------+--------------+------------+-------------+
|patient_id|admission_date|discharge_date|   diagnosis|hospital_bill|
+----------+--------------+--------------+------------+-------------+
|         1|    01-01-2022|    2022-01-10|   Pneumonia|       5000.0|
|         2|    02-05-2022|    2022-02-09|Appendicitis|       7000.0|
+----------+--------------+--------------+------------+-------------+
only showing top 2 rows



In [16]:
hosp_df3 = hosp_df2.withColumn("admission_date_1" ,to_date("admission_date","mm-dd-yyyy")).withColumn("discharge_date_1" ,to_date("discharge_date","yyyy-mm-dd"))

In [17]:
hosp_df3.show()

+----------+--------------+--------------+-------------+-------------+----------------+----------------+
|patient_id|admission_date|discharge_date|    diagnosis|hospital_bill|admission_date_1|discharge_date_1|
+----------+--------------+--------------+-------------+-------------+----------------+----------------+
|         1|    01-01-2022|    2022-01-10|    Pneumonia|       5000.0|      2022-01-01|      2022-01-10|
|         2|    02-05-2022|    2022-02-09| Appendicitis|       7000.0|      2022-01-05|      2022-01-09|
|         3|    03-12-2022|    2022-03-18|Fractured Arm|       3500.0|      2022-01-12|      2022-01-18|
|         4|    04-02-2022|    2022-04-08| Heart Attack|      15000.0|      2022-01-02|      2022-01-08|
|         5|    05-05-2022|    2022-05-07|    Influenza|       2500.0|      2022-01-05|      2022-01-07|
|         6|    06-10-2022|    2022-06-15| Appendicitis|       8000.0|      2022-01-10|      2022-01-15|
|         7|    07-20-2022|    2022-07-25|    Pneumonia

In [18]:
hosp_df3.printSchema()

root
 |-- patient_id: integer (nullable = true)
 |-- admission_date: string (nullable = true)
 |-- discharge_date: string (nullable = true)
 |-- diagnosis: string (nullable = true)
 |-- hospital_bill: double (nullable = true)
 |-- admission_date_1: date (nullable = true)
 |-- discharge_date_1: date (nullable = true)



In [26]:
hosp_df4 = hosp_df3.selectExpr("*","discharge_date_1 - admission_date_1 as duration_of_stay")

In [27]:
hosp_df4.show(5)

+----------+--------------+--------------+-------------+-------------+----------------+----------------+----------------+
|patient_id|admission_date|discharge_date|    diagnosis|hospital_bill|admission_date_1|discharge_date_1|duration_of_stay|
+----------+--------------+--------------+-------------+-------------+----------------+----------------+----------------+
|         1|    01-01-2022|    2022-01-10|    Pneumonia|       5000.0|      2022-01-01|      2022-01-10|          9 days|
|         2|    02-05-2022|    2022-02-09| Appendicitis|       7000.0|      2022-01-05|      2022-01-09|          4 days|
|         3|    03-12-2022|    2022-03-18|Fractured Arm|       3500.0|      2022-01-12|      2022-01-18|          6 days|
|         4|    04-02-2022|    2022-04-08| Heart Attack|      15000.0|      2022-01-02|      2022-01-08|          6 days|
|         5|    05-05-2022|    2022-05-07|    Influenza|       2500.0|      2022-01-05|      2022-01-07|          2 days|
+----------+------------

In [35]:
hosp_df4_1 = hosp_df3.withColumn("duration_of_stay",expr("datediff(discharge_date_1, admission_date_1)"))

In [36]:
hosp_df4.printSchema()

root
 |-- patient_id: integer (nullable = true)
 |-- admission_date: string (nullable = true)
 |-- discharge_date: string (nullable = true)
 |-- diagnosis: string (nullable = true)
 |-- hospital_bill: double (nullable = true)
 |-- admission_date_1: date (nullable = true)
 |-- discharge_date_1: date (nullable = true)
 |-- duration_of_stay: interval (nullable = true)



In [37]:
hosp_df4_1.printSchema()

root
 |-- patient_id: integer (nullable = true)
 |-- admission_date: string (nullable = true)
 |-- discharge_date: string (nullable = true)
 |-- diagnosis: string (nullable = true)
 |-- hospital_bill: double (nullable = true)
 |-- admission_date_1: date (nullable = true)
 |-- discharge_date_1: date (nullable = true)
 |-- duration_of_stay: integer (nullable = true)



In [38]:
hosp_df4_1.show(3)

+----------+--------------+--------------+-------------+-------------+----------------+----------------+----------------+
|patient_id|admission_date|discharge_date|    diagnosis|hospital_bill|admission_date_1|discharge_date_1|duration_of_stay|
+----------+--------------+--------------+-------------+-------------+----------------+----------------+----------------+
|         1|    01-01-2022|    2022-01-10|    Pneumonia|       5000.0|      2022-01-01|      2022-01-10|               9|
|         2|    02-05-2022|    2022-02-09| Appendicitis|       7000.0|      2022-01-05|      2022-01-09|               4|
|         3|    03-12-2022|    2022-03-18|Fractured Arm|       3500.0|      2022-01-12|      2022-01-18|               6|
+----------+--------------+--------------+-------------+-------------+----------------+----------------+----------------+
only showing top 3 rows



In [28]:
hosp_df5 = hosp_df4.selectExpr("*","CASE WHEN diagnosis = 'Heart Attack' THEN hospital_bill * 1.5 WHEN diagnosis='Appendicitis' THEN hospital_bill * 1.2 ELSE hospital_bill END AS adjusted_total_cost")

In [29]:
hosp_df5.show(10)

+----------+--------------+--------------+-------------+-------------+----------------+----------------+----------------+-------------------+
|patient_id|admission_date|discharge_date|    diagnosis|hospital_bill|admission_date_1|discharge_date_1|duration_of_stay|adjusted_total_cost|
+----------+--------------+--------------+-------------+-------------+----------------+----------------+----------------+-------------------+
|         1|    01-01-2022|    2022-01-10|    Pneumonia|       5000.0|      2022-01-01|      2022-01-10|          9 days|             5000.0|
|         2|    02-05-2022|    2022-02-09| Appendicitis|       7000.0|      2022-01-05|      2022-01-09|          4 days|             8400.0|
|         3|    03-12-2022|    2022-03-18|Fractured Arm|       3500.0|      2022-01-12|      2022-01-18|          6 days|             3500.0|
|         4|    04-02-2022|    2022-04-08| Heart Attack|      15000.0|      2022-01-02|      2022-01-08|          6 days|            22500.0|
|     

In [30]:
hosp_df5.select("patient_id", "diagnosis", "hospital_bill", "adjusted_total_cost" ).show(4)

+----------+-------------+-------------+-------------------+
|patient_id|    diagnosis|hospital_bill|adjusted_total_cost|
+----------+-------------+-------------+-------------------+
|         1|    Pneumonia|       5000.0|             5000.0|
|         2| Appendicitis|       7000.0|             8400.0|
|         3|Fractured Arm|       3500.0|             3500.0|
|         4| Heart Attack|      15000.0|            22500.0|
+----------+-------------+-------------+-------------------+
only showing top 4 rows

